# 02 — Train M0 (baseline) then M1 (curriculum)
M0 must finish first: M1's config asserts against M0's `run_meta.json` (checkpoint hash + step count). Debug on the 0.5B config before spending T4/L4 hours on 1.5B/3B (Part 2 practical plan).

In [2]:

# --- Self-contained Colab bootstrap (Part 0) ---
# Every notebook does this independently: Colab does not guarantee a new
# notebook tab reuses a previous notebook's VM, so nothing installed or
# cloned in another notebook can be assumed to exist here. This is
# idempotent -- re-running it (e.g. because you ARE still on the same
# runtime) just no-ops the clone and re-pulls latest.
import os, subprocess, shutil
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
if not os.path.exists('/content/verilog-slm'):
    !git clone {REPO} /content/verilog-slm
%cd /content/verilog-slm
!git pull

CKPT = '/content/drive/MyDrive/verilog-slm/checkpoints'
LOGS = '/content/drive/MyDrive/verilog-slm/logs'
os.makedirs(CKPT, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
!ln -sfn {CKPT} artifacts_drive_ckpt
!ln -sfn {LOGS} artifacts_drive_logs

# artifacts/corpus.jsonl and data/eval/*.jsonl are built once in
# 01_data.ipynb but land on THAT run's local (ephemeral) VM disk -- a
# different notebook tab is not guaranteed to reuse the same VM, so
# without this they'd be missing here (this is exactly the
# FileNotFoundError: 'artifacts/corpus.jsonl' failure mode). 01_data.ipynb
# copies them to this same Drive folder after building them; restore them
# here if this fresh VM doesn't have them locally yet. No-ops harmlessly
# if the local copy already exists or Drive doesn't have one yet.
DATA = '/content/drive/MyDrive/verilog-slm/data'
os.makedirs(DATA, exist_ok=True)
os.makedirs('data/eval', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)
for rel_path in ['artifacts/corpus.jsonl', 'data/eval/verilogeval_v2.jsonl', 'data/eval/rtllm_v2.jsonl']:
    drive_path = f"{DATA}/{os.path.basename(rel_path)}"
    if os.path.exists(drive_path) and not os.path.exists(rel_path):
        shutil.copy(drive_path, rel_path)
        print(f"restored {rel_path} from Drive")

Mounted at /content/drive
Cloning into '/content/verilog-slm'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 176 (delta 87), reused 127 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 143.68 KiB | 8.98 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/verilog-slm
Already up to date.
restored artifacts/corpus.jsonl from Drive
restored data/eval/verilogeval_v2.jsonl from Drive
restored data/eval/rtllm_v2.jsonl from Drive


In [3]:
# Pinned deps (Part 0 hygiene: Colab silently upgrades packages between sessions)
!pip install -q -r requirements.txt -r requirements-train.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 kB 11.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.3/342.3 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.5 MB/s eta 0:00:00


In [4]:
# requirements-train.txt deliberately doesn't pin torch (Colab ships one
# already matched to the VM's CUDA driver) -- log what's actually here
# instead, per Part 0's "pin every dependency" hygiene.
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())

torch: 2.11.0+cu128 | CUDA: 12.8 | GPU available: True


In [5]:
import os
# RTL toolchain: iverilog is required (compile+simulate). yosys and verible
# are optional -- the harness soft-gates on them (see docs/industry_standards.md)
# but install them here so the synthesis/lint stages actually run instead of
# being recorded as 'skipped'.
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null

# verible's release asset filename embeds a version string that changes
# every release (verible-v0.0-NNNN-gHASH-linux-static-x86_64.tar.gz), so a
# fixed "latest/download/<literal-name>" URL goes stale -- resolve the
# actual asset URL via the GitHub API instead. Chained as one shell
# command (not separate `!` lines) so the VERIBLE_URL variable survives
# across the pipe/curl/tar steps -- each `!` line is its own subprocess,
# so a bare shell variable assignment on its own line would silently be
# treated as Python and never reach bash at all.
!VERIBLE_URL=$(curl -s https://api.github.com/repos/chipsalliance/verible/releases/latest \
  | grep -o '"browser_download_url": *"[^"]*linux-static-x86_64.tar.gz"' \
  | head -1 | cut -d'"' -f4) && echo "resolved verible URL: $VERIBLE_URL" && \
  curl -sL "$VERIBLE_URL" -o /tmp/verible.tar.gz && \
  mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1

os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
resolved verible URL: https://github.com/chipsalliance/verible/releases/download/v0.0-4219-g3275ab72/verible-v0.0-4219-g3275ab72-linux-static-x86_64.tar.gz
Icarus Verilog version 12.0 (stable) ()
Unable to get version from "/usr/lib/x86_64-linux-gnu/ivl/ivlpp -V"
Unable to get version from "/usr/lib/x86_64-linux-gnu/ivl/ivl -V -C"/tmp/ivrlh2bec8319" -C"/usr/lib/x86_64-linux-gnu/ivl/vvp.conf""
Yosys 0.33 (git sha1 2584903a060)
Version	v0.0-4219-g3275ab72
Commit-Timestamp	2026-09-16T07:19:22Z
Built	2026-09-16T07:21:53Z


In [6]:
# Record the GPU model at the start of every run -- required for the
# per-GPU-hour metric (Part 0 non-negotiable hygiene) to mean anything.
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-eec9aa4b-53f1-32e2-d1a1-ef431fc0befe)


## Debug pass (0.5B) -- minutes, not hours
Smoke-tests the actual training code path (QLoRA load, LoRA attach, optimizer/scheduler, batch construction, checkpoint save) end to end before committing real GPU-hours to M0. Uses `configs/debug_0.5b.yaml` -- a separate config (not a hand-edited copy of the real one) so this run can never be mistaken for M0 or accidentally feed M1's `assert_against`.

In [ ]:
!python -m src.train.sft --config configs/debug_0.5b.yaml

In [ ]:
# Sanity check: confirm it actually trained (loss logged, checkpoint saved,
# run_meta written) rather than silently no-oping somewhere.
import json
meta = json.load(open('artifacts/debug_0.5b/run_meta.json'))
print('completed steps:', meta['total_steps'])
print('train GPU-hours:', meta['train_gpu_hours'])
print('checkpoint exists:', __import__('os').path.exists('artifacts/debug_0.5b/final'))
log_lines = open('artifacts/debug_0.5b/train_log.jsonl').readlines()
print('loss log entries:', len(log_lines))
print('first logged loss:', json.loads(log_lines[0])['loss'])
print('last logged loss:', json.loads(log_lines[-1])['loss'])

## Memory-fit check on the REAL M0 model (1.5B, real seq_len=2048)
The 0.5B debug pass above only proved the code path works -- it used a much shorter seq_len to guarantee it fit, so it does NOT prove `base.yaml`'s revised batch/seq_len settings actually fit the real 1.5B model. This runs 5 real steps on the real model to confirm before committing to the full multi-hour run.

In [7]:
!python -m src.train.sft --config configs/m0_memcheck.yaml

[sft] GPU: GPU 0: Tesla T4 (UUID: GPU-eec9aa4b-53f1-32e2-d1a1-ef431fc0befe)
config.json: 100% 660/660 [00:00<00:00, 3.42MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 22.1MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 114MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 124MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 133MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   6% 186M/3.09G [00:01<00:13, 215MB/s, 13.9MB/s  ]
model.safetensors: downloading bytes:   7% 227M/3.09G [00:01<00:13, 210MB/s, 20.0MB/s  ]
model.safetensors: downloading bytes:   9% 269M/3.09G [00:02<00:15, 178MB/s, 23.4MB/s  ]
model.safetensors: downloading bytes:  14% 431M/3.09G [00:02<00:09, 285MB/s, 35.3MB/s  ]
model.safetensors: downloading bytes:  59% 1.83G/3.09G [00:07<00:05, 245MB/s,  127MB/s  ]
model.safetensors: downloading bytes:  66% 2.04G/3.09G [00:09<00:09, 110MB/s,  125MB/s  ]
model.safetensors: downloading bytes:  80% 2.47G

### If the check above passed but was very slow (per-step time implies the full run would take dozens of hours), test a less conservative batch size before committing -- `per_device_batch_size=1` avoids OOM but is GPU-inefficient (one example at a time).

In [8]:
!python -m src.train.sft --config configs/m0_memcheck_bs2.yaml

[sft] GPU: GPU 0: Tesla T4 (UUID: GPU-eec9aa4b-53f1-32e2-d1a1-ef431fc0befe)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:03<00:00, 86.42it/s] 
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[sft] step 0/5 loss=1.3930 ema=1.3930
[sft] done. 5 steps in 0.07 GPU-hours. Artifacts -> artifacts/m0_memcheck_bs2


In [10]:
!ls -la artifacts_drive_ckpt/m0/checkpoint-200/

total 144323
drwx------ 2 root root      4096 Sep 12 15:54 .
drwx------ 4 root root      4096 Sep 12 15:54 ..
-rw------- 1 root root      1165 Sep 12 15:54 adapter_config.json
-rw------- 1 root root 147770496 Sep 12 15:54 adapter_model.safetensors
-rw------- 1 root root      5218 Sep 12 15:54 README.md


In [11]:
!rm -rf artifacts/m0 artifacts_drive_ckpt/m0

## M0 -- flat SFT baseline
The real run. Record: total optimizer steps, wall-clock, GPU model, peak VRAM (Part 5).

In [ ]:
!python -m src.train.sft --config configs/m0_baseline.yaml 2>&1 | tee artifacts_drive_logs/m0_stdout.log


[sft] GPU: GPU 0: Tesla T4 (UUID: GPU-eec9aa4b-53f1-32e2-d1a1-ef431fc0befe)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 92.11it/s] 
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[sft] step 0/2997 loss=1.4678 ema=1.4678
[sft] step 10/2997 loss=1.2104 ema=1.4499
[sft] step 20/2997 loss=0.9682 ema=1.3851
[sft] step 30/2997 loss=0.6255 ema=1.2548
[sft] step 40/2997 loss=0.567

## Diagnostic pass (Part 6) -- produces the table M1's reweighting needs
Run before M1. See notebooks/03_diagnose.ipynb for the full breakdown; the minimum needed here is `artifacts/m0_diagnostic.json`.

In [ ]:
!python -m src.infer.generate --adapter artifacts/m0/final --split probe \
  --n 5 --temperature 0.8 --top_p 0.95 --out artifacts/m0_probe_gens.jsonl
!python -m src.eval.diagnose --gens artifacts/m0_probe_gens.jsonl --out artifacts/m0_diagnostic.json

## M1 -- curriculum SFT, same step budget as M0
`assert_matches_m0` inside sft.py fails loudly if the base checkpoint or step count diverge from M0 -- this is the guarantee that keeps the ablation clean (Part 8).

In [ ]:
!python -m src.train.sft --config configs/m1_curriculum.yaml 2>&1 | tee artifacts_drive_logs/m1_stdout.log
!cp -r artifacts/m1 artifacts_drive_ckpt/m1

In [ ]:
# Plot realised category histogram: M1 actually saw vs. M0 (uniform) --
# the direct evidence the curriculum did what it was designed to do (Part 7).
import json, matplotlib.pyplot as plt
m1_meta = json.load(open('artifacts/m1/run_meta.json'))
hist = m1_meta['realised_histogram']['construct']
plt.bar(hist.keys(), hist.values())
plt.xticks(rotation=60, ha='right')
plt.title('M1 realised construct-tag exposure over training')
plt.tight_layout()
plt.savefig('artifacts/m1_realised_histogram.png')
plt.show()